# Domain 5: Context Management & Reliability
## Task Statement 5.2: Design Effective Escalation and Ambiguity Resolution Patterns
**Certification Weight**: 17% Exam Weight | **Standard**: Claude 4.5 & Extended Thinking

## 🎯 Exam Objective
Understand how to implement deterministic escalation routing for **explicit human requests**, resolve **multi-match query ambiguities** without guessing, and preserve conflicting statistics with source provenance.

---

## 🏗️ Architecture: Escalation and Ambiguity Router

```mermaid
flowchart TD
    UserQuery["Incoming User Query / Action"] --> AmbiguityCheck{"Evaluate Intent & State"}
    
    AmbiguityCheck -->|"User says: 'Let me talk to a person'"| ImmediateEscalate["Deterministic Gate:<br>Immediate Human Escalation<br>(Never attempt to deflect)"]
    
    AmbiguityCheck -->|"Search returns 2+ accounts"| Disambiguate["Ambiguity Protocol:<br>Prompt user to select between accounts<br>(Do NOT guess or pick randomly)"]
    
    AmbiguityCheck -->|"Action exceeds agent policy"| PolicyEscalate["Escalate with Structured Handoff<br>(Reason, context facts, actions taken)"]
    
    AmbiguityCheck -->|"Normal operational request"| Autonomous["Continue Autonomous Execution"]
```

---

## ⚠️ Critical Exam Distinctions & Anti-Patterns

| Situation | Correct Agent Behavior | Exam Trap to Reject |
|---|---|---|
| **Explicit Human Request** | **Escalate immediately**: Respect user demand without further deflection, sales pitches, or repeated queries. | **TRAP**: Trying to convince the user to keep talking to the AI or attempting 2 more times to resolve it autonomously. |
| **Multiple Database Matches** | **Disambiguation prompt**: Present the conflicting records to the user and request specific clarifying criteria. | **TRAP**: Arbitrarily picking the first record or guessing which account belongs to the user. |
| **Conflicting Source Data** | **Preserve both perspectives**: Quote each conflicting source with timestamp and methodology in the synthesis. | **TRAP**: Averaging numbers together or silently picking one source without telling the user about the conflict. |

---

## 💡 Architectural Decision Matrix

| Trigger Event | Routing Destination | Payload Structure |
|---|---|---|
| **"Human Agent Please"** | Tier 2 Human Support | Full Case Facts + Escalation Reason |
| **Policy Limit Exceeded ($500+)** | Supervisor Approval Queue | Structured Handoff Bundle |
| **Multiple Search Results Found** | Interactive User Disambiguation | Multiple-choice options |
| **Standard Supported Task** | Autonomous Agent Loop | Tool Execution Results |

---

---

### 🧠 Claude 4.5 Extended Thinking & Parameter Standards (October 2026)
- **Model Standard**: Use `claude-sonnet-4-5` or `claude-haiku-4-5`.
- **Temperature Omission**: When `thinking` is enabled (`thinking={"type": "enabled", "budget_tokens": ...}`), **do not provide `temperature`**, `top_p`, or `top_k`. The Anthropic API rejects custom sampling values with an HTTP `400 Bad Request` error.
- **Thinking Continuity**: In multi-turn loops, assistant reasoning blocks (`type: "thinking"`) must be returned **unmodified with their cryptographic signature** in subsequent turns. Stripping thinking blocks causes an API `400 Bad Request` error.
- **Case Facts Preservation**: Immutable transactional data (Order IDs, customer accounts, amounts) must be maintained in a dedicated Case Facts block outside conversation summaries.

In [ ]:
# Step 1: Environment & Setup
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(" Environment initialized for TS 5.2 Escalation & Provenance.")

### 🚦 Step 2: Escalation Evaluator & Provenance Synthesizer
We implement the escalation decision engine and conflicting source synthesizer.

In [ ]:
# Step 2: Escalation & Provenance Logic
@dataclass
class EscalationDecision:
    escalate: bool
    target: str  # 'human_supervisor' | 'user_clarification' | 'continue_automated'
    reason: str
    clarification_prompt: Optional[str] = None


@dataclass
class SourceClaim:
    metric_name: str
    value: str
    source_name: str
    published_date: str
    methodology: str


def evaluate_escalation_rules(user_message: str, matching_records_count: int, has_matching_policy: bool) -> EscalationDecision:
    """Evaluates user message and state against the architectural escalation matrix."""
    lower = user_message.lower()
    
    # 1. Customer demands human
    if any(phrase in lower for phrase in ["speak to a person", "human agent", "talk to someone", "representative"]):
        return EscalationDecision(
            escalate=True,
            target="human_supervisor",
            reason="Customer explicitly requested human assistance."
        )

    # 2. Ambiguity tie (multiple matching records)
    if matching_records_count > 1:
        return EscalationDecision(
            escalate=False,
            target="user_clarification",
            reason="Ambiguity tie: Multiple records match customer input.",
            clarification_prompt="Multiple accounts match your information. Please provide your Account ID or Postal Code."
        )

    # 3. Policy gap
    if not has_matching_policy:
        return EscalationDecision(
            escalate=True,
            target="human_supervisor",
            reason="Unhandled policy gap: No automated policy matches this scenario."
        )

    return EscalationDecision(escalate=False, target="continue_automated", reason="Standard automated handling.")


def synthesize_conflicting_sources(claims: List[SourceClaim]) -> Dict[str, Any]:
    """Synthesizes claims without averaging contradictory findings."""
    metric_name = claims[0].metric_name if claims else "Unknown"
    summary_lines = [
        f"Synthesized findings for '{metric_name}':",
        "NOTE: Contradictory findings detected between authoritative sources. Values are explicitly preserved:",
    ]
    for c in claims:
        summary_lines.append(f"- {c.source_name} ({c.published_date}): {c.value} via {c.methodology}")

    return {
        "metric": metric_name,
        "has_contested_findings": len(claims) > 1,
        "claims_with_provenance": [c.__dict__ for c in claims],
        "synthesis_summary": "\n".join(summary_lines),
    }

print(" Escalation rules and Provenance synthesizer ready.")

### 🧪 Step 3: Interactive Simulation
1. Customer says "Please let me speak to a person" -> Immediate human escalation.
2. Ambiguous query with 2 matching records -> Prompts user for clarification.
3. Two sources disagree on market metrics -> Both preserved without averaging.

In [ ]:
# Step 3: Simulation
# Case A: Human demand
dec_a = evaluate_escalation_rules("Please let me speak to a person right now.", 1, True)
print("Case A (Human Demand):", dec_a)

# Case B: Ambiguity tie (2 records found)
dec_b = evaluate_escalation_rules("Track my recent order", 2, True)
print("\nCase B (Ambiguity Tie):", dec_b)

# Case C: Conflicting sources synthesis
claims = [
    SourceClaim("Active Users", "100 Million", "Survey Inc", "2026-01-15", "Self-Reported Sample"),
    SourceClaim("Active Users", "120 Million", "Financial Q4 Filing", "2026-02-01", "Telemetry Audit"),
]
synth_res = synthesize_conflicting_sources(claims)
print("\nCase C (Provenance Synthesis):\n", synth_res["synthesis_summary"])

### 🔬 Step 4: Verification Assertions
We assert that:
1. Human demand escalates to `human_supervisor`.
2. Multiple records triggers `user_clarification`.
3. Conflicting claims are preserved with source citations without averaging.

In [ ]:
# Step 4: Verification Checks
assert dec_a.escalate is True
assert dec_a.target == "human_supervisor"

assert dec_b.escalate is False
assert dec_b.target == "user_clarification"
assert "Multiple accounts match" in dec_b.clarification_prompt

assert synth_res["has_contested_findings"] is True
assert len(synth_res["claims_with_provenance"]) == 2
assert "Survey Inc" in synth_res["synthesis_summary"]
assert "Financial Q4 Filing" in synth_res["synthesis_summary"]

print(" All verification assertions passed! Escalation and provenance verified.")

### 💡 CCAR-F Exam Golden Rules
- **Rule 1**: Immediately escalate to a human when the user explicitly requests one.
- **Rule 2**: When multiple records match (ambiguity), **clarify with the user**; do not guess or escalate.
- **Rule 3**: Never average conflicting numbers; preserve both claims with explicit source provenance.